# 03 · 속성별 층화와 평균의 함정

같은 page score에서 micro와 macro 집계가 어떻게 다른 결론을 만들 수 있는지 보여준다. 공식 dataset 결과는 사용하지 않는다.

**학습 목표**: token-weighted micro 평균과 문서 속성별 macro 평균이 hard-case를 다르게 드러내는 이유를 설명한다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 표준 라이브러리 `collections`만 사용하며 외부 패키지는 없다.

In [ ]:
# defaultdict(list)는 새로운 group key의 빈 목록 초기화를 자동화한다.
from collections import defaultdict

samples = [
    {'type': 'paper', 'difficulty': 'easy', 'tokens': 1000, 'score': .98},
    {'type': 'paper', 'difficulty': 'easy', 'tokens': 900, 'score': .96},
    {'type': 'newspaper', 'difficulty': 'hard', 'tokens': 120, 'score': .62},
    {'type': 'note', 'difficulty': 'hard', 'tokens': 80, 'score': .70},
]

def micro_average(rows):
    return sum(r['score'] * r['tokens'] for r in rows) / sum(r['tokens'] for r in rows)

def macro_by(rows, key):
    groups = defaultdict(list)
    for row in rows:
        groups[row[key]].append(row['score'])
    means = {group: sum(values) / len(values) for group, values in groups.items()}
    return means, sum(means.values()) / len(means)


In [ ]:
micro = micro_average(samples)
by_type, type_macro = macro_by(samples, 'type')
by_difficulty, difficulty_macro = macro_by(samples, 'difficulty')
print('token-weighted micro:', round(micro, 3))
print('type means:', by_type, 'macro:', round(type_macro, 3))
print('difficulty means:', by_difficulty, 'macro:', round(difficulty_macro, 3))
assert micro > difficulty_macro


큰 easy page가 많은 micro 평균은 hard failure를 가린다. benchmark report에는 평균 정의, group별 표본 수와 confidence interval을 함께 남기는 편이 안전하다.